# Synthetic lateral-movement sensitivity experiment v2

This notebook adds sparse aggregate-flow overlays to NF-CSE-CIC-IDS2018 Day 2 and evaluates the existing champion checkpoints without retraining or threshold adjustment. It creates **counterfactual NetFlow-like records, not packets**, and does not prove successful authentication, remote execution, or lateral movement.

The fixed experiment contains 54 attack scenarios: SMB/RPC, RDP, and SSH × three discovered targets × 5/15/30-minute horizons × direct valid-account or authentication-attempt paths. Every attack is paired with three unrelated endpoint-permuted controls. Numeric donor features are not inherently malicious; the experiment asks whether fixed models react differently when the same service-flow characteristics are attached to the documented compromised pivot and its preceding history.

In [ ]:
# SETUP - Run only when the repository is not already in /content.
!git clone -b feat/fair-comparison-experiments https://github.com/tatipar/temporalgnn-nids.git

In [ ]:
from pathlib import Path
import sys

from google.colab import drive
drive.mount('/content/drive')
!pip -q install torch-geometric

REPO_ROOT = Path('/content/temporalgnn-nids')
if not REPO_ROOT.exists():
    raise FileNotFoundError('Clone or upload temporalgnn-nids to /content/temporalgnn-nids first.')
sys.path.insert(0, str(REPO_ROOT / 'code/python'))

In [ ]:
# Paths and immutable experiment settings
import torch

ANALYSIS_ROOT = Path('/content/drive/MyDrive/nids-mitre')
DAY2_CSV = ANALYSIS_ROOT / 'data/cicids2018-v3/cicids2018v3_thu0103.csv'
DAY2_GRAPH_ROOT = ANALYSIS_ROOT / 'dataset_processed_thu0103'
RECOVERED_MAP = DAY2_GRAPH_ROOT / 'ip_map_day2_recovered.pkl'
SCALER_PATH = ANALYSIS_ROOT / 'dataset_processed/scaler_wed2802.pkl'
RESULTS_DIR = ANALYSIS_ROOT / 'results_earlystopping'
CORRECTED_CAMPAIGN_DIR = ANALYSIS_ROOT / 'analysis_mitre_corrected_v1'
OUTPUT_DIR = ANALYSIS_ROOT / 'synthetic_lateral_movement_v2'

PIVOT_IP = '172.31.69.13'
ATTACKER_IP = '13.58.225.34'
GENERATION_SEED = 20260804
CONTROL_SOURCES_PER_ATTACK = 3
RUN_DIAGNOSTIC_ABLATIONS = True
OVERWRITE_OVERLAYS = False
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Device:', DEVICE)
print('Output:', OUTPUT_DIR)

In [ ]:
# Imports
import gc
import glob
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from utils.evaluation import gather_metrics, apply_1sd_rule
from utils.models import (
    SimpleMLP, E_GraphSAGE, EdgeGRU_Baseline_NoX,
    StaticGNN_Identity, ST_GNN_Identity,
)
from utils.synthetic_lm import (
    SyntheticLMError, apply_diagnostic_gate, attach_corrected_campaign_ground_truth,
    audit_ip_map_usage, build_paired_diagnostics, build_raw_timing_index,
    build_scenario_matrix, count_trainable_parameters, create_matched_controls,
    create_synthetic_flows, donor_support_report, enrich_base_availability,
    evaluate_model_scenarios, load_ip_map, load_relevant_day2_flows, select_targets,
    summarize_diagnostic_ablations, summarize_experiment, summarize_paired_diagnostics,
    validate_overlay_layout, validate_synthetic_flows, write_sparse_overlays,
)

## 1. Preflight and scenario construction

Targets must have exact-port Discovery evidence from the infected host. Donor support must contain either at least 20 exact-port internal TCP flows or at least 50 flows in the same model port category. Donor labels are retained for provenance; a donor flow being benign does not prevent its numeric vector from representing a plausible remote-service session.

In [ ]:
required_paths = [
    DAY2_CSV, DAY2_GRAPH_ROOT / 'test2', RECOVERED_MAP, SCALER_PATH,
    RESULTS_DIR, CORRECTED_CAMPAIGN_DIR,
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required paths:\n- ' + '\n- '.join(missing))

ip_to_id, id_to_ip = load_ip_map(RECOVERED_MAP)
scaler = joblib.load(SCALER_PATH)
relevant_flows, global_start, global_end = load_relevant_day2_flows(
    DAY2_CSV, pivot_ip=PIVOT_IP, attacker_ip=ATTACKER_IP
)
targets = select_targets(relevant_flows, ip_to_id, pivot_ip=PIVOT_IP)
audit = audit_ip_map_usage(
    DAY2_GRAPH_ROOT, ip_to_id, [PIVOT_IP, ATTACKER_IP, *targets['Target_IP']]
)
support = donor_support_report(relevant_flows)
attack_manifest = build_scenario_matrix(targets, global_start, global_end, pivot_ip=PIVOT_IP)
targets.to_csv(OUTPUT_DIR / 'selected_targets.csv', index=False)
support.to_csv(OUTPUT_DIR / 'donor_support_report.csv', index=False)
pd.DataFrame([audit]).to_csv(OUTPUT_DIR / 'ip_map_audit.csv', index=False)

print(audit)
display(targets)
display(support)
print('Attack scenarios:', len(attack_manifest))

## 2. Generate donor-based flows and full matched controls

Every one of the 54 attacks receives three controls. A control retains the linked flow's 20 numeric features, port, protocol, target, timestamp, and donor provenance while replacing the compromised source with a campaign-unrelated mapped internal endpoint. Selection prefers benign-only source hosts, but Day 2's broad native attack interval can leave none; in that case it falls back to the lowest native attack-label-rate sources and records that fact as provenance. These are constructed controls, not verified benign sessions. The original graph tensors remain the prefix of every modified graph; only affected windows are written.

In [ ]:
synthetic_attack_flows = create_synthetic_flows(
    relevant_flows, attack_manifest, global_start,
    attacker_ip=ATTACKER_IP, seed=GENERATION_SEED,
)
control_flows, control_manifest = create_matched_controls(
    synthetic_attack_flows, attack_manifest, relevant_flows, ip_to_id,
    controls_per_attack=CONTROL_SOURCES_PER_ATTACK,
)
all_flows = pd.concat([synthetic_attack_flows, control_flows], ignore_index=True)
all_manifest = pd.concat([attack_manifest, control_manifest], ignore_index=True)
control_source_columns = [
    'control_source_rank', 'control_source_ip', 'control_source_selection_tier',
    'control_source_retained_source_flows',
    'control_source_retained_destination_flows',
    'control_source_retained_benign_source_flows',
    'control_source_retained_native_attack_source_flows',
    'control_source_retained_native_attack_rate',
]
control_source_report = (
    control_manifest[control_source_columns]
    .drop_duplicates('control_source_ip')
    .sort_values('control_source_rank')
)
control_source_report.to_csv(OUTPUT_DIR / 'control_source_report.csv', index=False)
validation_issues = validate_synthetic_flows(all_flows, all_manifest, raise_on_error=False)
if not validation_issues.empty:
    display(validation_issues)
    raise SyntheticLMError('Synthetic-flow validation failed; overlays were not written.')

write_report_path = OUTPUT_DIR / 'overlay_write_report.csv'
if OVERWRITE_OVERLAYS or not write_report_path.exists():
    write_report = write_sparse_overlays(
        DAY2_GRAPH_ROOT, OUTPUT_DIR, all_flows, all_manifest, ip_to_id, scaler
    )
else:
    write_report = pd.read_csv(write_report_path)
    provenance_columns = [
        'Synthetic_Event_ID', 'Scenario_ID', 'Donor_Row_ID',
        'Linked_Attack_Event_ID', 'IPV4_SRC_ADDR', 'window_id',
    ]
    saved_flows = pd.read_csv(
        OUTPUT_DIR / 'synthetic_flows.csv', usecols=provenance_columns
    ).sort_values('Synthetic_Event_ID').reset_index(drop=True)
    expected_flows = all_flows[provenance_columns].sort_values(
        'Synthetic_Event_ID'
    ).reset_index(drop=True)
    pd.testing.assert_frame_equal(saved_flows, expected_flows, check_dtype=False)
    saved_scenarios = set(pd.read_csv(OUTPUT_DIR / 'scenario_manifest.csv')['scenario_id'])
    assert saved_scenarios == set(all_manifest['scenario_id']), 'Stored overlays are stale'

expected_controls = len(attack_manifest) * CONTROL_SOURCES_PER_ATTACK
assert len(attack_manifest) == 54
assert len(control_manifest) == expected_controls
assert control_manifest.groupby('linked_attack_scenario').size().eq(
    CONTROL_SOURCES_PER_ATTACK
).all()
assert write_report['Scenario_ID'].nunique() == len(all_manifest)
layout_report = validate_overlay_layout(OUTPUT_DIR, all_flows)
assert not layout_report[['Missing_Windows', 'Unexpected_Windows']].to_numpy().any()

donor_provenance = (
    synthetic_attack_flows.groupby(['Event_Role', 'Donor_Original_Attack'], as_index=False)
    .size().rename(columns={'size': 'Synthetic_Flows'})
)
donor_provenance.to_csv(OUTPUT_DIR / 'donor_label_provenance.csv', index=False)
display(control_source_report)
display(donor_provenance)
print('Attack overlays:', len(attack_manifest))
print('Control overlays:', len(control_manifest))

## 3. Load unchanged champion checkpoints and thresholds

No synthetic result participates in champion selection or threshold selection.

In [ ]:
MODEL_CONFIG = {
    'node_dim': 16, 'edge_dim': 32, 'hidden_dim': 32,
    'dropout': 0.2, 'output_bias_init': -2.9968,
}
NAME_MAP = {
    'SimpleMLP_BiasOn': 'Simple MLP',
    'EGraphSAGE_BiasOn': 'E-GraphSAGE',
    'EdgeGRU_NoX_BiasOn': 'Edge GRU',
    'StaticGNN_BiasOn_robust_Identity': 'Static GNN',
    'ST_GNN_BiasOn_robust_Identity_clone': 'ST-GNN',
}
MODEL_SPECS = {
    'Simple MLP': (SimpleMLP, 'SimpleMLP_BiasOn', False),
    'E-GraphSAGE': (E_GraphSAGE, 'EGraphSAGE_BiasOn', False),
    'Edge GRU': (EdgeGRU_Baseline_NoX, 'EdgeGRU_NoX_BiasOn', True),
    'Static GNN': (StaticGNN_Identity, 'StaticGNN_BiasOn_robust_Identity', False),
    'ST-GNN': (ST_GNN_Identity, 'ST_GNN_BiasOn_robust_Identity_clone', True),
}

df_metrics = gather_metrics(RESULTS_DIR / 'logs', NAME_MAP)
df_champions = apply_1sd_rule(df_metrics)

def load_champion(model_class, experiment_name):
    champion = df_champions[df_champions['Raw_Dir_Name'].eq(experiment_name)]
    if champion.empty:
        raise ValueError(f'No champion found for {experiment_name}')
    seed = int(champion['Seed'].iloc[0])
    threshold_path = RESULTS_DIR / 'logs' / experiment_name / f'thresholds_{experiment_name}.npz'
    threshold = float(np.load(threshold_path)[f'seed_{seed}'])
    metric_candidates = [
        RESULTS_DIR / 'logs' / experiment_name / f'run_metrics_{experiment_name}.csv',
        RESULTS_DIR / 'logs' / experiment_name / f'metrics_newth_{experiment_name}.csv',
    ]
    metrics_path = next(path for path in metric_candidates if path.exists())
    metrics = pd.read_csv(metrics_path)
    metrics.columns = metrics.columns.str.lower()
    run_column = 'run_id' if 'run_id' in metrics.columns else 'extra_run_id'
    selected = metrics[metrics['model_name'].str.contains(f'seed{seed}', na=False)]
    run_id = selected[run_column].iloc[0]
    checkpoints = glob.glob(str(RESULTS_DIR / 'saved_models' / experiment_name / f'{run_id}_*.pth'))
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint for {experiment_name}, run {run_id}')
    model = model_class(**MODEL_CONFIG).to(DEVICE)
    model.load_state_dict(torch.load(checkpoints[0], map_location=DEVICE))
    model.eval()
    return model, threshold, seed

display(df_champions[df_champions['Raw_Dir_Name'].isin(NAME_MAP)][
    ['Raw_Dir_Name', 'Seed', 'F1_Score']
] )

## 4. Fixed-model inference and diagnostic ablations

Each model processes the base sequence once. Temporal memory is cached immediately before each scenario's first changed window and restored for every independent overlay. EdgeGRU and ST-GNN additionally run two attack-only diagnostics: `reset_memory` removes all history before the short scenario interval, while `remove_focus_context` retains memory but removes original current-window edges incident to the pivot or target. The latter changes both local topology and available flow context and is not a pure causal topology estimate.

In [ ]:
raw_timing = build_raw_timing_index(
    DAY2_CSV, [PIVOT_IP, *targets['Target_IP'].tolist()], global_start
)

all_base_events = []
all_scenario_predictions = []
model_cost_rows = []
for model_name, (model_class, experiment_name, temporal) in MODEL_SPECS.items():
    print(f'\nEvaluating {model_name}...')
    model, threshold, champion_seed = load_champion(model_class, experiment_name)
    trainable_parameters = count_trainable_parameters(model)
    run_ablations = RUN_DIAGNOSTIC_ABLATIONS and temporal
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats(DEVICE)
    started = time.perf_counter()
    base_events, full_predictions = evaluate_model_scenarios(
        model=model, model_name=model_name, threshold=threshold,
        base_graph_root=DAY2_GRAPH_ROOT, overlay_root=OUTPUT_DIR,
        scenario_manifest=all_manifest, id_to_ip=id_to_ip, pivot_ip=PIVOT_IP,
        device=DEVICE, temporal=temporal, ablation_modes=('full',),
    )
    full_inference_seconds = time.perf_counter() - started
    full_peak_mib = (
        torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
        if torch.cuda.is_available() else np.nan
    )
    ablation_inference_seconds = 0.0
    ablation_peak_mib = np.nan
    predictions = full_predictions
    if run_ablations:
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats(DEVICE)
        ablation_started = time.perf_counter()
        _, ablation_predictions = evaluate_model_scenarios(
            model=model, model_name=model_name, threshold=threshold,
            base_graph_root=DAY2_GRAPH_ROOT, overlay_root=OUTPUT_DIR,
            scenario_manifest=attack_manifest, id_to_ip=id_to_ip, pivot_ip=PIVOT_IP,
            device=DEVICE, temporal=temporal,
            ablation_modes=('reset_memory', 'remove_focus_context'),
        )
        ablation_inference_seconds = time.perf_counter() - ablation_started
        ablation_peak_mib = (
            torch.cuda.max_memory_allocated(DEVICE) / (1024 ** 2)
            if torch.cuda.is_available() else np.nan
        )
        predictions = pd.concat([full_predictions, ablation_predictions], ignore_index=True)
    base_events = enrich_base_availability(base_events, raw_timing)
    base_events['Champion_Seed'] = champion_seed
    predictions['Champion_Seed'] = champion_seed
    base_events.to_csv(OUTPUT_DIR / f'base_events_{experiment_name}.csv', index=False)
    predictions.to_csv(OUTPUT_DIR / f'scenario_predictions_{experiment_name}.csv', index=False)
    all_base_events.append(base_events)
    all_scenario_predictions.append(predictions)
    model_cost_rows.append({
        'Model': model_name,
        'Trainable_Parameters': trainable_parameters,
        'Full_Inference_Seconds': full_inference_seconds,
        'Ablation_Inference_Seconds': ablation_inference_seconds,
        'Full_Synthetic_Flows_Evaluated': int(
            (predictions['Diagnostic_Ablation'].eq('full') & predictions['Is_Synthetic']).sum()
        ),
        'Diagnostic_Ablation_Modes': (
            'reset_memory,remove_focus_context' if run_ablations else 'none'
        ),
        'Full_CUDA_Peak_Memory_MiB': full_peak_mib,
        'Ablation_CUDA_Peak_Memory_MiB': ablation_peak_mib,
    })
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

base_events = pd.concat(all_base_events, ignore_index=True)
scenario_predictions = pd.concat(all_scenario_predictions, ignore_index=True)
base_events = attach_corrected_campaign_ground_truth(base_events, CORRECTED_CAMPAIGN_DIR)
model_cost_report = pd.DataFrame(model_cost_rows)
base_events.to_csv(OUTPUT_DIR / 'base_events_all_models.csv', index=False)
scenario_predictions.to_csv(OUTPUT_DIR / 'scenario_predictions_all_models.csv', index=False)
model_cost_report.to_csv(OUTPUT_DIR / 'model_cost_report.csv', index=False)
display(model_cost_report)

## 5. End-to-end lead time and revised diagnostic gate

The v2 gate requires ≥70% LM scenario coverage, ≥70% same-pivot precursor coverage, ≥70% joint precursor-plus-LM coverage, median operational lead ≥5 minutes, joint coverage of at least two targets in at least two protocols, a graph or temporal model, and ≥10 percentage points scenario-macro advantage over all matched controls. The breadth metric now requires both warning and LM detection.

In [ ]:
attack_summary, control_summary = summarize_experiment(
    scenario_predictions, base_events, all_manifest
)
paired = build_paired_diagnostics(scenario_predictions, all_flows)
paired_overall = summarize_paired_diagnostics(paired)
paired_by_protocol = summarize_paired_diagnostics(paired, strata=('Protocol_Mechanism',))
paired_by_horizon = summarize_paired_diagnostics(paired, strata=('Horizon_Minutes',))
paired_by_access_path = summarize_paired_diagnostics(paired, strata=('Access_Path',))
paired_by_donor_label = summarize_paired_diagnostics(
    paired, strata=('Donor_Original_Attack',)
)

scenario_breakdown = (
    attack_summary.groupby(
        ['Model', 'Protocol', 'Horizon_Minutes', 'Access_Path'], as_index=False
    ).agg(
        Scenarios=('Scenario_ID', 'nunique'),
        LM_Coverage=('Synthetic_LM_Detected', 'mean'),
        Warning_Coverage=('Precursor_Warning', 'mean'),
        End_To_End_Coverage=('End_To_End_Detected', 'mean'),
        Median_Operational_Lead_Minutes=('Operational_Lead_Minutes', 'median'),
    )
)
precursor_provenance = (
    attack_summary.groupby(['Model', 'Precursor_Warning_Source'], as_index=False)
    .size().rename(columns={'size': 'Scenarios'})
)
gate = apply_diagnostic_gate(attack_summary, control_summary)

expected_pairs = (
    synthetic_attack_flows['Event_Role'].eq('synthetic_lm').sum()
    * CONTROL_SOURCES_PER_ATTACK * len(MODEL_SPECS)
)
assert len(paired) == expected_pairs
mlp_pairs = paired[paired['Model'].eq('Simple MLP')]
assert np.allclose(
    mlp_pairs['Linked_Probability'], mlp_pairs['Control_Probability'], atol=1e-7
), 'MLP endpoint-permutation invariant failed'

attack_summary.to_csv(OUTPUT_DIR / 'attack_scenario_summary.csv', index=False)
control_summary.to_csv(OUTPUT_DIR / 'matched_control_summary.csv', index=False)
paired.to_csv(OUTPUT_DIR / 'paired_flow_diagnostics.csv', index=False)
paired_overall.to_csv(OUTPUT_DIR / 'paired_overall_summary.csv', index=False)
paired_by_protocol.to_csv(OUTPUT_DIR / 'paired_by_protocol.csv', index=False)
paired_by_horizon.to_csv(OUTPUT_DIR / 'paired_by_horizon.csv', index=False)
paired_by_access_path.to_csv(OUTPUT_DIR / 'paired_by_access_path.csv', index=False)
paired_by_donor_label.to_csv(OUTPUT_DIR / 'paired_by_donor_label.csv', index=False)
scenario_breakdown.to_csv(OUTPUT_DIR / 'scenario_breakdown.csv', index=False)
precursor_provenance.to_csv(OUTPUT_DIR / 'precursor_warning_provenance.csv', index=False)
gate.to_csv(OUTPUT_DIR / 'diagnostic_gate_v2.csv', index=False)

display(gate)
display(paired_overall)
display(paired_by_protocol)
display(precursor_provenance)
if gate['Passes_Robust_Diagnostic_Gate'].any():
    print('CONTINUE: at least one fixed model passes the revised synthetic diagnostic gate.')
else:
    print('STOP: no fixed model passes the revised synthetic diagnostic gate.')

## 6. Temporal-memory and current-context sensitivity

A drop under `reset_memory` indicates dependence on pre-scenario temporal state. A drop under `remove_focus_context` indicates dependence on current-window local context. Because the latter removes both edges and their flow contributions, and because checkpoints were not trained under either intervention, these are sensitivity diagnostics rather than proof that GAT topology causally improves detection.

In [ ]:
ablation_summary = summarize_diagnostic_ablations(
    scenario_predictions, all_manifest
)
ablation_summary.to_csv(OUTPUT_DIR / 'diagnostic_ablation_summary.csv', index=False)
display(ablation_summary)

In [ ]:
# Compact result plots
sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 3, figsize=(19, 5))
gate.set_index('Model')[[
    'LM_Coverage', 'Warning_Coverage', 'End_To_End_Coverage'
]].plot.bar(ax=axes[0], ylim=(0, 1))
sns.barplot(
    data=gate, x='Model', y='Scenario_Macro_Control_Advantage', ax=axes[1]
)
sns.barplot(
    data=gate, x='Model', y='Median_Operational_Lead_Minutes', ax=axes[2]
)
axes[0].set_title('Synthetic scenario coverage')
axes[1].axhline(0.10, color='red', linestyle='--', label='gate threshold')
axes[1].set_title('Scenario-macro linked/control advantage')
axes[1].legend()
axes[2].set_title('Median operational precursor lead')
for axis in axes[1:]:
    axis.tick_params(axis='x', rotation=30)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'synthetic_lm_v2_summary.png', dpi=180, bbox_inches='tight')
plt.show()

## Interpretation limits

- Donor vectors can originate from benign or attack-labelled traffic; their synthetic label comes from the assumed scenario, not from an intrinsic malicious distribution.
- Controls are endpoint-permuted counterfactuals, not verified legitimate administrator sessions.
- The 54 scenarios reuse one real campaign prefix, nine Discovery anchors, and deterministic donor pools; they are not 54 independent captured attacks.
- `Assumed_Success` is constructed scenario ground truth, not success inferred from packet or host telemetry.
- The revised gate supports or rejects further analysis of fixed-model sensitivity. It does not estimate real-world lateral-movement recall, precision, or prevention probability.